<a href="https://colab.research.google.com/github/DaniilKuzma/ai-assistant/blob/main_multihybrid/notebooks/main_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
push = subprocess.run(
    ["git", "push", "--force", auth_url, "HEAD:main_multihybrid"],
    capture_output=True,
    text=True,
)

print(push.stdout.replace(TOKEN, "***TOKEN***").replace(TOKEN_ESCAPED, "***TOKEN***"))
print(push.stderr.replace(TOKEN, "***TOKEN***").replace(TOKEN_ESCAPED, "***TOKEN***"))
print("RETURN CODE:", push.returncode)


To https://github.com/DaniilKuzma/ai-assistant.git
 + 50f00bd...7d135d2 HEAD -> main_multihybrid (forced update)

RETURN CODE: 0


In [4]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/DaniilKuzma/ai-assistant.git"
BRANCH = "main_multihybrid"
PROJECT_DIR = Path("/content/ai-assistant")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    os.chdir(PROJECT_DIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(PROJECT_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("cwd:", Path.cwd())
print("src exists:", (PROJECT_DIR / "src").exists())

cwd: /content/ai-assistant
src exists: True


In [30]:
from pathlib import Path
import os
import subprocess

os.chdir("/content/ai-assistant")

subprocess.run(["git", "status"], check=True)

subprocess.run(["git", "config", "user.name", "DaniilKuzma"], check=True)
subprocess.run(["git", "config", "user.email", "kuznetcovdanil62rus@mail.ru"], check=True)

subprocess.run(["git", "add", "-A"], check=True)

status = subprocess.run(
    ["git", "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
)

if status.stdout.strip():
    subprocess.run(["git", "commit", "-m", "Update Colab outputs"], check=True)
    subprocess.run(["git", "push", "origin", "main_multihybrid"], check=True)
else:
    print("No changes to commit.")

No changes to commit.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
from datetime import datetime

drive.mount("/content/drive")

os.chdir("/content/ai-assistant")

run_id = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
drive_run_dir = Path("/content/drive/MyDrive/ai-assistant-runs") / run_id
drive_latest_dir = Path("/content/drive/MyDrive/ai-assistant-runs/latest")

drive_run_dir.mkdir(parents=True, exist_ok=True)
drive_latest_dir.mkdir(parents=True, exist_ok=True)

subprocess.run(["rsync", "-a", "models/", str(drive_run_dir / "models/")], check=True)
subprocess.run(["rsync", "-a", "reports/", str(drive_run_dir / "reports/")], check=True)

subprocess.run(["rsync", "-a", "models/", str(drive_latest_dir / "models/")], check=True)
subprocess.run(["rsync", "-a", "reports/", str(drive_latest_dir / "reports/")], check=True)

print("Saved run:", drive_run_dir)
print("Updated latest:", drive_latest_dir)

# Russian Edit Corrector Pipeline

Top-to-bottom entrypoint for config loading, data preparation, synthetic generation, labels, training scaffold, evaluation, reports, and manual examples.

In [5]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src').exists() else cwd.parent
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import load_config

config_path = project_root / 'configs' / 'config.yaml'
config = load_config(config_path)
config['project']['name']

'russian-edit-corrector'

In [ ]:
import sys
import torch

env = {
    'python': sys.version.split()[0],
    'cuda_available': torch.cuda.is_available(),
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
}
env

{'python': '3.10.12',
 'cuda_available': True,
 'device': 'NVIDIA GeForce RTX 4070'}

In [ ]:
import importlib
import src.data.full_dataset_builder as fdb

importlib.reload(fdb)

dataset_build = fdb.build_dataset_from_config(config, force=True)
dataset_build

Loading clean corpus cache: data/raw/clean_corpus_sentences.txt.gz
Loaded 105569 clean sentences from cache


{'status': 'built',
 'path': 'data/processed/correction_dataset.csv.gz',
 'manifest_path': 'reports/dataset_manifest.json',
 'total': 450000,
 'composition': {'clean': 45000, 'synthetic': 403403, 'real': 1597},
 'splits': {'test': 22500, 'val': 22501, 'train': 404999},
 'clean_corpus': {'count': 105569,
  'cache_path': 'data/raw/clean_corpus_sentences.txt.gz',
  'source_counts': {'cache': 105569}}}

In [ ]:
import pandas as pd

dataset_frame = pd.read_csv(config['data']['processed_train_path'])
examples = dataset_frame.head(6).to_dict('records')
dataset_frame.shape, dataset_frame['split'].value_counts().to_dict(), examples[:2]


In [ ]:
from src.alignment.aligner import Aligner

aligner = Aligner()
[(row['source'], aligner.align(row['source'], row['target']).is_supported) for row in examples]


[('врядли это сложный результат для документа 254813', True),
 ('кое как работает корпус 33069 но результат важен', True),
 ('кто нибудь проверит раздел 336868 если будет время', True),
 ('Сегодня важный день потому что готов раздел 330084', True),
 ('Я незнаю что делать с словарь 348190', True),
 ('Мы проверяем что то важное в разделе 42775', True)]

In [9]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [15]:
from src.training.train import train

train_result = train(config_path)
train_result

Building training features:   0%|          | 0/10000 [00:00<?, ?it/s]

Training setup: 10000 examples, 157 micro-steps/epoch, 157 optimizer steps/epoch, accumulation=1


Training epoch:   0%|          | 0/157 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from src.inference.corrector import Corrector
from src.inference.model_corrector import TrainedModelCorrector
from src.evaluation.metrics import compute_metrics

if 'examples' not in globals():
    import pandas as pd
    dataset_frame = pd.read_csv(config['data']['processed_train_path'])
    examples = dataset_frame.head(6).to_dict('records')

adapter_dir = project_root / config['paths']['adapter_output_dir']
heads_path = project_root / config['paths']['heads_output_dir'] / 'heads.pt'
use_trained_corrector = bool(train_result.get('model_training_ran')) and adapter_dir.exists() and heads_path.exists()
model_corrector = TrainedModelCorrector.from_config(config) if use_trained_corrector else Corrector()
corrector_kind = 'trained_model' if use_trained_corrector else 'rule_fallback'
rows = []
for row in examples:
    rows.append({
        'source': row['source'],
        'target': row['target'],
        'prediction': model_corrector.correct(row['source']).corrected_text,
        'is_clean': bool(row['is_clean']),
    })
corrector_kind, compute_metrics(rows)


('trained_model',
 {'exact_match': 0.5,
  'dirty_improved_rate': 0.5,
  'dirty_worse_rate': 0.5,
  'clean_overcorrection_rate': 0.0,
  'edit_precision': 1.0,
  'edit_recall': 0.8888888888888888,
  'edit_f1': 0.9411764705882353,
  'spelling_precision': 1.0,
  'spelling_recall': 0.8571428571428571,
  'spelling_f1': 0.923076923076923,
  'punctuation_precision': 1.0,
  'punctuation_recall': 1.0,
  'punctuation_f1': 1.0})

In [ ]:
if 'model_corrector' not in globals():
    from src.inference.model_corrector import TrainedModelCorrector
    model_corrector = TrainedModelCorrector.from_config(config)
    corrector_kind = 'trained_model'

manual = ['Я незнаю что делать', 'сегодня что то произошло', 'Я люблю этот дом']
corrector_kind, [(text, model_corrector.correct(text).corrected_text) for text in manual]

('trained_model',
 [('Я незнаю что делать', 'Я незнаю, что делать.'),
  ('сегодня что то произошло', 'Сегодня, что-то произошло.'),
  ('Я люблю этот дом', 'Я люблю этот дом.')])